# AI Assignment 2: Cybersecurity Turn-Based Game

## Overview
- **`game.py`**: Implements the `Host` and `GameState` classes (network representation, host statuses, legal action generation, state transitions, terminal detection, and utility calculation).
- **`algorithms.py`**: Implements Minimax, Depth-Limited Minimax (`depth = 3`), and Alpha-Beta Pruning.
- **`main.ipynb`**: Demonstrates **Task 1** (three test cases for legal moves, patching, and winning) and **Task 2** (algorithm comparisons and experimental results).

## Task 1: Model the Game & Deliverable Test Cases
First, we import `Host` and `GameState` from `game.py` and verify the initial network topology (10 hosts `H0`–`H9`, where `H0` starts compromised and `H5` holds the crown jewels database worth 100).

In [4]:
from game import *

# Initialize a new game and inspect the network
initial_state = GameState()
print("Initial State:", initial_state)
print("\nNetwork Hosts:")
for name, host in initial_state.hosts.items():
    print(f"  {host} -> connected to {host.connected_to}")

Initial State: GameState(plies used=0, current player=Attacker, defender budget=10, zero_day available=True, target exfiltrated=False)

Network Hosts:
  H0(value=0, compromised=True, patch=False, isolated=False) -> connected to ['H1', 'H2']
  H1(value=10, compromised=False, patch=False, isolated=False) -> connected to ['H0', 'H3', 'H4']
  H2(value=15, compromised=False, patch=False, isolated=False) -> connected to ['H0', 'H4']
  H3(value=30, compromised=False, patch=False, isolated=False) -> connected to ['H1', 'H5', 'H6']
  H4(value=10, compromised=False, patch=False, isolated=False) -> connected to ['H1', 'H2', 'H6']
  H5(value=100, compromised=False, patch=False, isolated=False) -> connected to ['H3', 'H7']
  H6(value=60, compromised=False, patch=False, isolated=False) -> connected to ['H3', 'H4', 'H7']
  H7(value=25, compromised=False, patch=False, isolated=False) -> connected to ['H5', 'H6', 'H8']
  H8(value=20, compromised=False, patch=False, isolated=False) -> connected to ['H7'

### Test Case 1: Legal Moves, Isolation & Worked Example (Section 2.6)
This test case demonstrates:
1. Initial legal moves for the Attacker from `H0`.
2. Executing the exact sequence from **Section 2.6 Worked Example**:
   - Attacker: `("Exploit", "H1")`
   - Defender: `("Patch", "H3")` (cost 2)
   - Attacker: `("Exploit", "H4")` (adjacent to compromised `H1`)
   - Defender: `("Isolate", "H6")` (cost 8, permanently removing all connections to `H6`)
3. Verifying that total defence cost spent is 10 (so only `("Pass", None)` remains legal for the Defender) and that the utility matches $U = -(10 + 10) - 10 = -30$.

In [8]:
state1 = GameState()

# 1. Check initial legal actions at Ply 0 (Attacker)
print("PLY 0 - INITIAL STATE")
print("Current state:", state1)
print("Legal actions:", state1.get_legal_actions())

# 2. Run the Section 2.6 Worked Example moves
state1.apply_action(("Exploit", "H1"))
print("\nPLY 1 - AFTER ATTACKER ACTION: Exploit H1")
print("Current state:", state1)
print("Changed host:", state1.hosts["H1"])
print("Legal actions:", state1.get_legal_actions())

state1.apply_action(("Patch", "H3"))
print("\nPLY 2 - AFTER DEFENDER ACTION: Patch H3")
print("Current state:", state1)
print("Changed host:", state1.hosts["H3"])
print("Legal actions:", state1.get_legal_actions())

state1.apply_action(("Exploit", "H4"))
print("\nPLY 3 - AFTER ATTACKER ACTION: Exploit H4")
print("Current state:", state1)
print("Changed host:", state1.hosts["H4"])
print("Legal actions:", state1.get_legal_actions())

state1.apply_action(("Isolate", "H6"))
print("\nPLY 4 - AFTER DEFENDER ACTION: Isolate H6")
print("Current state:", state1)
print("Changed host:", state1.hosts["H6"])
print("Legal actions:", state1.get_legal_actions())

# 3. Verify that once budget is 0, Defender can only Pass
state1.apply_action(("Pass", None))
print("\nPLY 5 - AFTER ATTACKER ACTION: Pass")
print("Current state:", state1)
print("Legal actions:", state1.get_legal_actions())

print("\nDefender utility:", state1.utility(), "(Expected: -30)")


PLY 0 - INITIAL STATE
Current state: GameState(plies used=0, current player=Attacker, defender budget=10, zero_day available=True, target exfiltrated=False)
Legal actions: [('Exploit', 'H1'), ('Exploit', 'H2'), ('Pass', None)]

PLY 1 - AFTER ATTACKER ACTION: Exploit H1
Current state: GameState(plies used=1, current player=Defender, defender budget=10, zero_day available=True, target exfiltrated=False)
Changed host: H1(value=10, compromised=True, patch=False, isolated=False)
Legal actions: [('Isolate', 'H1'), ('Restore', 'H1'), ('Patch', 'H2'), ('Isolate', 'H2'), ('Patch', 'H3'), ('Isolate', 'H3'), ('Patch', 'H4'), ('Isolate', 'H4'), ('Patch', 'H5'), ('Isolate', 'H5'), ('Patch', 'H6'), ('Isolate', 'H6'), ('Patch', 'H7'), ('Isolate', 'H7'), ('Patch', 'H8'), ('Isolate', 'H8'), ('Patch', 'H9'), ('Isolate', 'H9'), ('Pass', None)]

PLY 2 - AFTER DEFENDER ACTION: Patch H3
Current state: GameState(plies used=2, current player=Attacker, defender budget=8, zero_day available=True, target exfiltr

### Test Case 2: Patching, Zero-Day Bypass, and Host Restoration
This test case demonstrates the following sequence:
1. **Ply 0:** The initial Attacker state and legal actions.
2. **Ply 1:** Attacker uses `("Exploit", "H1")`.
3. **Ply 2:** Defender uses **`Patch(H3)`** (cost 2), preventing a normal `("Exploit", "H3")`.
4. **Ply 3:** Attacker uses **`ZeroDay(H3)`** to compromise the patched `H3` once, consuming `attacker_has_zero_day`.
5. **Ply 4:** Defender uses **`Restore(H3)`** (cost 5), cleaning `H3` while leaving it patched.
6. After Ply 4, `H3` can no longer be exploited or zero-dayed because the zero-day has been used.

In [12]:
state2 = GameState()

print("PLY 0 - INITIAL STATE")
print("Current state:", state2)
print("Legal actions:", state2.get_legal_actions())

# Ply 1: Attacker compromises H1
state2.apply_action(("Exploit", "H1"))
print("\nPLY 1 - AFTER ATTACKER ACTION: Exploit H1")
print("Current state:", state2)
print("Changed host:", state2.hosts["H1"])
print("Legal actions:", state2.get_legal_actions())

# Ply 2: Defender patches H3 (cost 2, budget -> 8)
state2.apply_action(("Patch", "H3"))
print("\nPLY 2 - AFTER DEFENDER ACTION: Patch H3")
print("Current state:", state2)
print("Changed host:", state2.hosts["H3"])
print("Legal actions:", state2.get_legal_actions())

# Ply 3: Attacker uses ZeroDay on patched H3
state2.apply_action(("ZeroDay", "H3"))
print("\nPLY 3 - AFTER ATTACKER ACTION: ZeroDay H3")
print("Current state:", state2)
print("Changed host:", state2.hosts["H3"])
print("Legal actions:", state2.get_legal_actions())

# Ply 4: Defender restores H3 (cost 5, budget -> 3) -> H3 becomes clean and patched
state2.apply_action(("Restore", "H3"))
print("\nPLY 4 - AFTER DEFENDER ACTION: Restore H3")
print("Current state:", state2)
print("Changed host:", state2.hosts["H3"])
print("Legal actions:", state2.get_legal_actions())

# Verification: Attacker can no longer attack H3 (patched and no Zero-Day remaining)
print("\nPLY 4 VERIFICATION - H3 REMAINS PROTECTED")
print("Current state:", state2)
print("Legal actions:", state2.get_legal_actions())

PLY 0 - INITIAL STATE
Current state: GameState(plies used=0, current player=Attacker, defender budget=10, zero_day available=True, target exfiltrated=False)
Legal actions: [('Exploit', 'H1'), ('Exploit', 'H2'), ('Pass', None)]

PLY 1 - AFTER ATTACKER ACTION: Exploit H1
Current state: GameState(plies used=1, current player=Defender, defender budget=10, zero_day available=True, target exfiltrated=False)
Changed host: H1(value=10, compromised=True, patch=False, isolated=False)
Legal actions: [('Isolate', 'H1'), ('Restore', 'H1'), ('Patch', 'H2'), ('Isolate', 'H2'), ('Patch', 'H3'), ('Isolate', 'H3'), ('Patch', 'H4'), ('Isolate', 'H4'), ('Patch', 'H5'), ('Isolate', 'H5'), ('Patch', 'H6'), ('Isolate', 'H6'), ('Patch', 'H7'), ('Isolate', 'H7'), ('Patch', 'H8'), ('Isolate', 'H8'), ('Patch', 'H9'), ('Isolate', 'H9'), ('Pass', None)]

PLY 2 - AFTER DEFENDER ACTION: Patch H3
Current state: GameState(plies used=2, current player=Attacker, defender budget=8, zero_day available=True, target exfiltr

### Test Case 3: Winning & Terminal Conditions (Exfiltration vs. 10-Ply Limit)
This test case demonstrates both ways the game ends:
- **Scenario 3A (Attacker Win via Early Exfiltration):** The Attacker compromises a path to `H5` (`H0 -> H1 -> H3 -> H5`) and executes `("Exfiltrate", None)` on Ply 7, ending the game immediately with the $-100$ exfiltration penalty.
- **Scenario 3B (Defender Survival across 10 Plies):** The Defender isolates `H5` (`("Isolate", "H5")`, cost 8), severing all paths to the crown jewels so the game ends at the 10-ply limit without exfiltration.

In [13]:
# --- Scenario 3A: Attacker reaches H5 and Exfiltrates on Ply 7 ---
game_exfil = GameState()
exfil_moves = [
    ("Exploit", "H1"),     # Ply 1 (Attacker): H1 compromised (val 10)
    ("Patch", "H2"),       # Ply 2 (Defender): H2 patched (cost 2)
    ("Exploit", "H3"),     # Ply 3 (Attacker): H3 compromised (val 30)
    ("Patch", "H4"),       # Ply 4 (Defender): H4 patched (cost 2)
    ("Exploit", "H5"),     # Ply 5 (Attacker): H5 compromised (val 100)
    ("Pass", None),        # Ply 6 (Defender): Pass (cost 0)
    ("Exfiltrate", None),  # Ply 7 (Attacker): Exfiltrate! Game ends immediately
]

for move in exfil_moves:
    game_exfil.apply_action(move)

print("=== Scenario 3A: Early Exfiltration ===")
print("Final State:", game_exfil)
print("is_terminal():", game_exfil.is_terminal(), "| Ended at ply:", game_exfil.ply_count)
print("Defender Utility U:", game_exfil.utility(), "(Expected: -(10+30+100) - 4 - 100 = -244)")
assert game_exfil.is_terminal() and game_exfil.exfiltrated and game_exfil.utility() == -244

# --- Scenario 3B: Defender isolates H5 and holds out for all 10 plies ---
game_hold = GameState()
hold_moves = [
    ("Exploit", "H1"),   # Ply 1 (Attacker): H1 compromised (10)
    ("Isolate", "H5"),   # Ply 2 (Defender): H5 isolated permanently (cost 8)
    ("Exploit", "H3"),   # Ply 3 (Attacker): H3 compromised (30), cannot reach isolated H5
    ("Patch", "H6"),     # Ply 4 (Defender): H6 patched (cost 2, budget -> 0)
    ("Exploit", "H4"),   # Ply 5 (Attacker): H4 compromised (10)
    ("Pass", None),      # Ply 6 (Defender)
    ("Exploit", "H2"),   # Ply 7 (Attacker): H2 compromised (15)
    ("Pass", None),      # Ply 8 (Defender)
    ("ZeroDay", "H6"),   # Ply 9 (Attacker): H6 compromised via ZeroDay (60)
    ("Pass", None),      # Ply 10 (Defender): Reaches 10-ply limit
]

for move in hold_moves:
    game_hold.apply_action(move)

print("\n=== Scenario 3B: 10-Ply Defender Hold ===")
print("Final State:", game_hold)
print("is_terminal():", game_hold.is_terminal(), "| Exfiltrated:", game_hold.exfiltrated)
print("Defender Utility U:", game_hold.utility(), "(Expected: -(10+30+10+15+60) - 10 - 0 = -135)")
assert game_hold.is_terminal() and not game_hold.exfiltrated and game_hold.utility() == -135

=== Scenario 3A: Early Exfiltration ===
Final State: GameState(plies used=7, current player=Defender, defender budget=6, zero_day available=True, target exfiltrated=True)
is_terminal(): True | Ended at ply: 7
Defender Utility U: -244 (Expected: -(10+30+100) - 4 - 100 = -244)

=== Scenario 3B: 10-Ply Defender Hold ===
Final State: GameState(plies used=10, current player=Attacker, defender budget=0, zero_day available=False, target exfiltrated=False)
is_terminal(): True | Exfiltrated: False
Defender Utility U: -135 (Expected: -(10+30+10+15+60) - 10 - 0 = -135)


## Task 2: Implement Minimax, Depth-Limited Minimax, and Alpha-Beta Pruning
In `algorithms.py`, we implement:
1. **`minimax(state, depth=None)`**: Runs standard full-depth Minimax when `depth=None`, and Depth-Limited Minimax when `depth=3`.
2. **`alpha_beta(state, depth=3, ...)`**: Runs Alpha-Beta Pruning (supports both `depth=3` and full-depth `depth=None`, with action ordering).

### Algorithm Comparison:
The experiment below compares full-depth Minimax, depth-limited Minimax (`depth=3`), full-depth Alpha-Beta, and depth-limited Alpha-Beta (`depth=3`).

Measured factors are runtime, expanded nodes, pruned nodes, selected action, and returned utility. Space complexity, optimality, and scalability are included as theoretical comparisons based on the search depth and branching factor.

The action-ordering strategy is the same for every search: `Exfiltrate` first, other active actions next, and `Pass` last. Minimax does not benefit from pruning, but it uses the same ordering so its result can be compared fairly with Alpha-Beta.

In [14]:
import time

from algorithms import alpha_beta, minimax
import algorithms


def run_algorithm(name, function, depth):
    # Reset counters so each experiment is measured independently.
    algorithms.expanded_nodes = 0
    algorithms.pruned_nodes = 0

    state = GameState()
    start_time = time.perf_counter()
    utility, action = function(state, depth=depth)
    elapsed = time.perf_counter() - start_time

    return {
        "Algorithm": name,
        "Depth": "Full" if depth is None else depth,
        "Time (s)": round(elapsed, 6),
        "Expanded nodes": algorithms.expanded_nodes,
        "Pruned nodes": algorithms.pruned_nodes,
        "Best utility": utility,
        "Best action": action,
    }


experiments = [
    ("Minimax", minimax, None),
    ("Depth-Limited Minimax", minimax, 3),
    ("Alpha-Beta", alpha_beta, None),
    ("Depth-Limited Alpha-Beta", alpha_beta, 3),
]

results = [run_algorithm(name, function, depth) for name, function, depth in experiments]

print("=== Task 2 Experimental Results ===")
for result in results:
    print("\n" + result["Algorithm"])
    print("  Search depth:", result["Depth"])
    print("  Runtime (seconds):", result["Time (s)"])
    print("  Expanded nodes:", result["Expanded nodes"])
    print("  Pruned nodes:", result["Pruned nodes"])
    print("  Best utility:", result["Best utility"])
    print("  Best action:", result["Best action"])

print("\n=== Task 2 Theoretical Comparison ===")
comparison = [
    ("Minimax", "O(b^d)", "Exact when full-depth", "Limited by exponential growth"),
    ("Depth-Limited Minimax", "O(b^d)", "Exact only to depth d; heuristic beyond", "More scalable because d is capped"),
    ("Alpha-Beta", "O(b^d) worst case; O(b^(d/2)) with strong ordering", "Exact when full-depth", "More scalable through pruning"),
]

print("Algorithm | Space | Optimality | Scalability")
for row in comparison:
    print(" | ".join(row))
print("\nAction ordering: Exfiltrate -> active actions -> Pass")


=== Task 2 Experimental Results ===

Minimax
  Search depth: Full
  Runtime (seconds): 190.194439
  Expanded nodes: 2402797
  Pruned nodes: 0
  Best utility: -44
  Best action: ('Exploit', 'H1')

Depth-Limited Minimax
  Search depth: 3
  Runtime (seconds): 0.005686
  Expanded nodes: 61
  Pruned nodes: 0
  Best utility: -20
  Best action: ('Exploit', 'H1')

Alpha-Beta
  Search depth: Full
  Runtime (seconds): 0.661536
  Expanded nodes: 14858
  Pruned nodes: 13519
  Best utility: -44
  Best action: ('Exploit', 'H1')

Depth-Limited Alpha-Beta
  Search depth: 3
  Runtime (seconds): 0.001356
  Expanded nodes: 28
  Pruned nodes: 81
  Best utility: -20
  Best action: ('Exploit', 'H1')

=== Task 2 Theoretical Comparison ===
Algorithm | Space | Optimality | Scalability
Minimax | O(b^d) | Exact when full-depth | Limited by exponential growth
Depth-Limited Minimax | O(b^d) | Exact only to depth d; heuristic beyond | More scalable because d is capped
Alpha-Beta | O(b^d) worst case; O(b^(d/2)) with